# v16 — Curriculum Learning (n_nouns 기반) + 자기완결형 노트북

**목적**: v14(ListwiseSoftmaxLoss)를 그대로 유지하면서, 학습 샘플을 보여주는 **순서**를 난이도 순으로
조정하는 curriculum learning을 단일 변수로 검증한다. K/loss/hard-negative 구성은 v14와 완전히 동일.

**난이도 정의**: EDA(§10-2, `idea.md`)에서 다중공선성 제거 후 유일하게 독립적인 정확도 예측 신호로
확정된 `n_nouns`(문장의 명사 개수)를 사용. n_nouns가 높은 샘플(모델이 80%+ 맞히는 구간)을 먼저,
낮은 샘플(30%대 구간, 텍스트 단서가 부족해 이미지에만 의존해야 하는 어려운 구간)을 뒤로 갈수록
비중을 늘리는 완만한 ramp-up 방식 — LR이 cosine decay로 후반부에 낮아지는 것과 정면충돌하지
않도록, 마지막 epoch에 어려운 것만 몰아넣지 않고 epoch1부터 소량씩 섞어서 점증시킨다.

**인프라 목적**: 여러 서버에 이 노트북 하나만 두면 모델/데이터 존재 확인 → (없으면) 다운로드 →
GPU 개수 자동 감지 → `accelerate.notebook_launcher`로 멀티 GPU 학습 → 추론 → 제출 파일 생성까지
전부 자기완결적으로 수행한다. 실행은 `jupyter nbconvert --to notebook --execute`로 헤드리스
실행하는 것을 권장(출력/로그가 노트북 파일 자체에 저장됨) — `run_notebook.sh` 참고.

## 1. 환경 체크

In [ ]:
import sys, os, subprocess, importlib

REQUIRED = ["torch", "transformers", "peft", "accelerate", "pandas", "PIL", "tqdm", "huggingface_hub"]
missing = []
for pkg in REQUIRED:
    name = "PIL" if pkg == "PIL" else pkg
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"누락된 패키지 설치 중: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
else:
    print("필요한 패키지 전부 설치되어 있음")

import torch
print(f"torch={torch.__version__}  cuda_available={torch.cuda.is_available()}")

# FlashAttention2 — 멀티이미지 어텐션 특성상 sdpa/eager 대비 속도 차이가 큼.
# 컴파일이 필요해 수 분 걸릴 수 있고 환경에 따라 실패할 수 있음 — 실패해도 sdpa로 자동 폴백되니 여기서 안 죽음.
try:
    import flash_attn
    print(f"flash-attn 이미 설치됨: {flash_attn.__version__}")
except ImportError:
    print("flash-attn 설치 시도 중... (컴파일 필요, 수 분 소요될 수 있음)")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
        import flash_attn
        print(f"flash-attn 설치 완료: {flash_attn.__version__}")
    except Exception as e:
        print(f"flash-attn 설치 실패 — sdpa로 폴백함(속도 저하 있을 수 있음): {e}")

## 2. Config

이 서버에서만 다르게 잡아야 하는 값(경로 등)은 이 셀에서만 수정하면 된다 — 나머지 셀은 전부 이
변수들을 참조한다(v8~v15의 `config.py`를 노트북 변수로 인라인).

In [ ]:
from pathlib import Path

# 전부 노트북 파일 기준 상대경로 — 어느 서버에 옮겨놔도 그 자리에서 그대로 동작
PROJECT_ROOT = Path(".")
DATA_DIR     = Path("./data/snuaichallenge_data")
MODEL_LOCAL  = Path("./models/Qwen3-VL-8B-Instruct")
MODEL_HF_ID  = "Qwen/Qwen3-VL-8B-Instruct"  # 로컬에 없으면 여기서 다운로드
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
LOG_DIR      = PROJECT_ROOT / "logs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGE_SIZE = 448
VAL_RATIO      = 0.05
SEED           = 42

# hard negative — v8/v14와 완전히 동일(K=7, d1..6 전 구간). curriculum 실험이라 이 구성은 안 건드림.
SAMPLE_COUNTS = {1: 2, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1}

LORA_R, LORA_ALPHA, LORA_DROPOUT = 128, 256, 0.05
LR = 5e-5
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
TRAIN_MINIBATCH = 8   # 2026-07-08 그리드서치+실제 accelerate 검증으로 확정된 이 하드웨어의 한계(K=7 기준)
INFER_BATCH_SIZE = 24

# ── curriculum 전용 파라미터 ──────────────────────────────────
# n_nouns tier: EDA §10-2 구간과 동일하게 3단계로 단순화
#   tier 0(easy, n_nouns>=9): val 80%+ 구간   tier 1(medium, 7~8): ~64%   tier 2(hard, 1~6): ~30~35%
N_NOUNS_EASY_MIN   = 9
N_NOUNS_MEDIUM_MIN = 7
# epoch별 tier 가중치 스케줄 (easy, medium, hard) — 합이 1일 필요는 없음(상대 비율로만 씀)
# epoch1부터 hard를 아예 안 보여주진 않되(완전 배제는 또 다른 위험), 비중을 서서히 늘림.
CURRICULUM_SCHEDULE = {
    1: (1.0, 0.8, 0.4),
    2: (1.0, 0.9, 0.6),
    3: (1.0, 1.0, 0.8),
    4: (1.0, 1.0, 1.0),   # 후반부는 균등 — LR이 이미 낮아진 마지막 epoch에 hard를 더 몰아주지 않음
    5: (1.0, 1.0, 1.0),
}

CKPT_NAME = "best_v16"
print("config 로드 완료")

## 3. GPU 개수 자동 감지

In [ ]:
import torch
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    raise RuntimeError("GPU가 감지되지 않았습니다. GPU 노드에서 실행해야 합니다.")
print(f"감지된 GPU 개수: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. 모델 가중치 확인/다운로드

로컬(`./models/...`)에 이미 완전하게 있으면 그대로 쓰고, 없으면 `!hf download`
셸 명령으로 **정확히 `./` 밑에만** 받는다 — `HF_HOME`도 `./`로 강제 리다이렉트해서 root 홈
디렉토리는 전혀 건드리지 않음 (Qwen3-VL-8B-Instruct는 공개 모델, gated 아님).

In [ ]:
import os
# Xet 백엔드(hf_xet)는 컴퓨트 노드 방화벽이 Xet CDN 도메인을 막아두면 타임아웃 없이 멈추는 경우가 있음
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 캐시/락 파일까지 전부 ./ 밑에만 쓰도록 강제 (root 홈 디렉토리 절대 사용 금지)
os.environ["HF_HOME"] = str((PROJECT_ROOT / ".cache" / "huggingface").resolve())
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

def model_is_complete():
    index_file = MODEL_LOCAL / "model.safetensors.index.json"
    if not index_file.exists():
        return False
    import json
    weight_map = json.loads(index_file.read_text())["weight_map"]
    return all((MODEL_LOCAL / f).exists() for f in set(weight_map.values()))

if model_is_complete():
    print(f"모델 이미 존재(무결성 확인됨): {MODEL_LOCAL}")
else:
    MODEL_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    # 이전에 중간에 끊긴 시도들이 안 지워진 lock을 남겨서 새 다운로드가 무한 대기하는 경우가 있음
    _stale_locks = list(MODEL_LOCAL.rglob("*.lock"))
    if _stale_locks:
        print(f"stale lock {len(_stale_locks)}개 정리: {[str(p) for p in _stale_locks]}")
        for _lf in _stale_locks:
            _lf.unlink(missing_ok=True)
    print(f"모델 다운로드 중 -> {MODEL_LOCAL} (hf download, ./ 밑에 직접 설치)")
    !hf download {MODEL_HF_ID} --local-dir {MODEL_LOCAL} --token {HF_TOKEN}
    if not model_is_complete():
        raise RuntimeError(f"다운로드 후에도 {MODEL_LOCAL}의 safetensors 샤드가 불완전합니다 — 위 로그 확인 필요")
    print(f"다운로드 완료: {MODEL_LOCAL}")

MODEL_PATH = str(MODEL_LOCAL)

## 5. 데이터 확인/다운로드 (Kaggle API)

Kaggle 토큰은 코드 안에 직접 박아두고, 실행 시점에 표준 경로 `~/.kaggle/access_token`에
자동으로 기록해서 `kagglehub`가 인증하도록 한다. 다운로드는 `!python3 -c "..."` 셸 명령으로 실행하고,
`KAGGLEHUB_CACHE`와 `output_dir`을 둘 다 `./` 밑으로 강제해서 root 홈 디렉토리는 절대 안 씀
(root 파티션이 작아서 거기 받으면 용량 부족으로 죽는 문제가 있었음).

In [ ]:
KAGGLE_TOKEN = "YOUR_KAGGLE_API_TOKEN_HERE"
_kdir = Path.home() / ".kaggle"
_kdir.mkdir(parents=True, exist_ok=True)
_ktok = _kdir / "access_token"
_ktok.write_text(KAGGLE_TOKEN)
os.chmod(_ktok, 0o600)
# kagglehub 캐시도 전부 ./ 밑으로 강제 (root 홈 디렉토리 용량 부족 방지)
os.environ["KAGGLEHUB_CACHE"] = str((PROJECT_ROOT / ".cache" / "kagglehub").resolve())

if (DATA_DIR / "train.csv").exists() and (DATA_DIR / "test.csv").exists():
    print(f"데이터 이미 존재: {DATA_DIR}")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"데이터 다운로드 중 -> {DATA_DIR} (kagglehub, ./ 밑에 직접 설치)")
    !python3 -c "import kagglehub; kagglehub.competition_download('snuaichallenge', output_dir='{DATA_DIR}')"
    if not (DATA_DIR / "train.csv").exists():
        candidates = list(DATA_DIR.rglob("train.csv"))
        if candidates:
            DATA_DIR = candidates[0].parent
        else:
            raise FileNotFoundError(f"{DATA_DIR} 아래에서 train.csv를 못 찾았습니다 — 위 다운로드 로그를 확인하세요.")
    print(f"최종 DATA_DIR = {DATA_DIR}")

## 6. n_nouns 계산 (curriculum 난이도 신호, 캐시 재사용)

이 노트북(=이 프로젝트 폴더) 안에 캐시가 있으면 재사용, 없으면 spaCy로 새로 계산해서
`./checkpoints/`에 저장 — 서버마다 독립적으로 캐시를 가짐(다른 프로젝트 폴더를 참조하지 않음).

In [ ]:
import pandas as pd
from collections import Counter

N_NOUNS_CACHE_CANDIDATES = [
    CKPT_DIR / "_n_nouns_lookup.csv",
]

def compute_n_nouns(train_df):
    for cand in N_NOUNS_CACHE_CANDIDATES:
        if cand.exists():
            print(f"n_nouns 캐시 재사용: {cand}")
            lookup = pd.read_csv(cand)
            if set(train_df["Id"]).issubset(set(lookup["Id"])):
                return lookup[["Id", "n_nouns"]]
    print("n_nouns 캐시 없음 — spaCy로 새로 계산 (전체 train, 몇 분 소요)")
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
        nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    n_nouns_list = []
    for doc in nlp.pipe(train_df["Sentence"].tolist(), batch_size=256):
        pos_counts = Counter(t.pos_ for t in doc if t.is_alpha)
        n_nouns_list.append(pos_counts.get("NOUN", 0) + pos_counts.get("PROPN", 0))
    out = pd.DataFrame({"Id": train_df["Id"], "n_nouns": n_nouns_list})
    out.to_csv(CKPT_DIR / "_n_nouns_lookup.csv", index=False)
    return out

train_csv_full = pd.read_csv(DATA_DIR / "train.csv")
n_nouns_df = compute_n_nouns(train_csv_full)
train_csv_full = train_csv_full.merge(n_nouns_df, on="Id", how="left")
print(train_csv_full["n_nouns"].describe())

## 7. Hard Negative / Dataset / Loss / Model 정의 (v14와 동일 로직, 외부 파일 의존 없이 인라인)

In [ ]:
import ast, random, time, csv
from itertools import permutations
from PIL import Image
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

ALL_PERMS = list(permutations([1, 2, 3, 4]))

def kendall_dist(p, q):
    rank = {v: i for i, v in enumerate(q)}
    arr = [rank[v] for v in p]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv

def _perms_by_dist(gt):
    by_dist = {}
    for p in ALL_PERMS:
        if p == gt:
            continue
        d = kendall_dist(p, gt)
        by_dist.setdefault(d, []).append(p)
    return by_dist

def sample_group(gt):
    """v8/v14와 완전히 동일: K=7, d1..6 전 구간 커버. curriculum은 '어떤 샘플을 방문하냐'만
    바꾸지 '그룹 안의 negative 구성'은 절대 안 건드림(단일 변수 원칙)."""
    by_dist = _perms_by_dist(gt)
    samples = [(list(gt), 0)]
    for dist, n_take in SAMPLE_COUNTS.items():
        pool = by_dist.get(dist, [])
        for p in random.sample(pool, min(n_take, len(pool))):
            samples.append((list(p), dist))
    return samples

PROMPT = (
    "Sentence: {sentence}\n\n"
    "These 4 frames are presented in this exact order.\n"
    "Please carefully examine the changes between consecutive frames.\n"
    "Is this the correct chronological order of events?\n"
    "Answer only with \"Yes\" or \"No\"."
)
SYSTEM = (
    "You are a temporal ordering assistant. "
    "Given video frames in a specific order and a caption, "
    "determine if the frames are in the correct chronological order."
)

def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = MAX_IMAGE_SIZE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def build_messages(images, sentence):
    content = []
    for i, img in enumerate(images, 1):
        content.append({"type": "text", "text": f"Frame {i}:"})
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": PROMPT.format(sentence=sentence)})
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}]

class GroupedTemporalDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = row["Id"]
        gt = tuple(ast.literal_eval(row["Answer"]))
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]
        images_list, sentences, labels, dists = [], [], [], []
        for perm, dist in sample_group(gt):
            inv = [0] * 4
            for inp_idx, t_pos in enumerate(perm):
                inv[t_pos - 1] = inp_idx
            imgs = [base_imgs[inv[t]] for t in range(4)]
            images_list.append(imgs)
            sentences.append(row["Sentence"])
            labels.append(1.0 if dist == 0 else 0.0)
            dists.append(int(dist))
        return {"images": images_list, "sentences": sentences, "labels": labels,
                "dists": dists, "group_size": len(images_list)}

def collate_fn(batch):
    return {
        "images": [b["images"] for b in batch],
        "sentences": [b["sentences"] for b in batch],
        "labels": [b["labels"] for b in batch],
        "dists": [b["dists"] for b in batch],
        "group_sizes": [b["group_size"] for b in batch],
    }

class ListwiseSoftmaxLoss:
    """v14와 동일: 1 positive + 7 negative를 8-way joint softmax cross-entropy로 취급."""
    def __init__(self):
        self.ema = torch.zeros(6)  # 로깅 호환용 placeholder
        self.step = 0

    def __call__(self, logits, dists, group_sizes):
        offset = 0
        losses = []
        neg_prob_by_dist = {d: [] for d in range(1, 7)}
        for gs in group_sizes:
            g_logits = logits[offset: offset + gs]
            g_dists = dists[offset: offset + gs]
            offset += gs
            pos_idxs = [i for i, d in enumerate(g_dists) if d == 0]
            if not pos_idxs:
                continue
            log_probs = torch.log_softmax(g_logits, dim=0)
            losses.append(-log_probs[pos_idxs[0]])
            probs = log_probs.exp().detach()
            for i, d in enumerate(g_dists):
                if d > 0:
                    neg_prob_by_dist[d].append(probs[i].item())
        total = torch.stack(losses).mean()
        per_dist_diag = {d: (sum(v) / len(v) if v else float("nan")) for d, v in neg_prob_by_dist.items()}
        self.step += 1
        return total, per_dist_diag, torch.full((6,), 1.0 / 6.0)

def get_model_class(model_path):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(model_path)
    mt = getattr(cfg, "model_type", "")
    if mt == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration
        return Qwen3VLForConditionalGeneration
    if mt == "qwen2_5_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration
        return Qwen2_5_VLForConditionalGeneration
    from transformers import Qwen2VLForConditionalGeneration
    return Qwen2VLForConditionalGeneration

def load_model_and_processor(resume_from=None):
    from transformers import AutoProcessor
    from peft import LoraConfig, PeftModel, get_peft_model, TaskType
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    ModelClass = get_model_class(MODEL_PATH)
    model = None
    for attn_impl in ("flash_attention_2", "sdpa", "eager"):
        try:
            model = ModelClass.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation=attn_impl)
            break
        except Exception:
            continue
    model.gradient_checkpointing_enable()
    if resume_from:
        model = PeftModel.from_pretrained(model, resume_from, is_trainable=True)
    else:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
        )
        model = get_peft_model(model, lora_config)
    return model, processor

def get_yes_no_token_ids(processor):
    tok = processor.tokenizer
    return tok.convert_tokens_to_ids(tok.tokenize("Yes"))[-1], tok.convert_tokens_to_ids(tok.tokenize("No"))[-1]

def forward_logit(model, inputs, yes_id, no_id):
    outputs = model(**inputs)
    last_logits = outputs.logits[:, -1, :].float()
    log_probs = torch.log_softmax(last_logits, dim=-1)
    score = log_probs[:, yes_id] - log_probs[:, no_id]
    return score.clamp(-100.0, 100.0)

print("핵심 로직 정의 완료 (hard_negative / dataset / loss / model)")

## 8. Curriculum Sampler

n_nouns tier별 가중치를 `CURRICULUM_SCHEDULE`(§2)에 따라 epoch마다 다르게 적용하는
`WeightedRandomSampler`를 매 epoch 새로 만든다. DDP 환경에서는 rank별로 독립적으로
가중 샘플링(replacement=True)하므로 별도 분산 조율 없이도 각 rank가 스스로 일관된
curriculum을 따름 — 다만 매 epoch 정확히 전체를 1회씩 커버하는 기존(v14) 방식과 달리
일부 샘플은 여러 번, 일부는 덜 보일 수 있음(가중 샘플링의 표준적 트레이드오프).

In [ ]:
import numpy as np

def assign_tier(n_nouns):
    if n_nouns >= N_NOUNS_EASY_MIN:
        return 0  # easy
    elif n_nouns >= N_NOUNS_MEDIUM_MIN:
        return 1  # medium
    else:
        return 2  # hard

def build_epoch_sampler(df_with_tier, epoch, dataset_len):
    w_easy, w_med, w_hard = CURRICULUM_SCHEDULE.get(epoch, (1.0, 1.0, 1.0))
    tier_w = np.array([w_easy, w_med, w_hard])
    sample_weights = tier_w[df_with_tier["tier"].values]
    return WeightedRandomSampler(sample_weights, num_samples=dataset_len, replacement=True)

print("curriculum sampler 정의 완료")
print("스케줄:", CURRICULUM_SCHEDULE)

## 9. Validation 함수

In [ ]:
def val_exact_match(model, processor, val_raw_df, yes_id, no_id, device):
    model.eval()
    correct, wrong_cases = 0, []
    for _, row in val_raw_df.iterrows():
        sid = row["Id"]
        sentence = row["Sentence"]
        gt = ast.literal_eval(row["Answer"])
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        def reorder(order):
            inv = [0] * 4
            for inp_idx, t_pos in enumerate(order):
                inv[t_pos - 1] = inp_idx
            return [base_imgs[inv[t]] for t in range(4)]

        texts, imgs_list = [], []
        for perm in ALL_PERMS:
            imgs = reorder(list(perm))
            msgs = build_messages(imgs, sentence)
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            texts.append(text)
            imgs_list.append(imgs)

        with torch.no_grad():
            inp = processor(text=texts, images=imgs_list, return_tensors="pt", padding=True).to(model.device)
            s = forward_logit(model, inp, yes_id, no_id)
        scores = [(s[i].item(), list(p)) for i, p in enumerate(ALL_PERMS)]
        best = max(scores, key=lambda x: x[0])[1]
        if best == gt:
            correct += 1
        else:
            diffs = [i for i in range(4) if best[i] != gt[i]]
            t = "adj_swap" if (len(diffs) == 2 and abs(diffs[0] - diffs[1]) == 1) else "other"
            wrong_cases.append(t)

    acc = correct / len(val_raw_df)
    adj = sum(1 for t in wrong_cases if t == "adj_swap")
    print(f"[Val] exact_match={acc:.4f} ({correct}/{len(val_raw_df)})  adj_swap_fail={adj}  other_fail={len(wrong_cases)-adj}")
    model.train()
    return acc

## 10. 학습 함수 (`accelerate.notebook_launcher`로 멀티 GPU 실행)

In [ ]:
def train_fn():
    from datetime import timedelta
    from torch.optim import AdamW
    from transformers import get_cosine_schedule_with_warmup
    from accelerate import Accelerator
    from accelerate.utils import InitProcessGroupKwargs

    pg_kwargs = InitProcessGroupKwargs(timeout=timedelta(days=2))
    accelerator = Accelerator(gradient_accumulation_steps=GRAD_ACCUM, kwargs_handlers=[pg_kwargs])
    device = accelerator.device
    is_main = accelerator.is_main_process

    torch.manual_seed(SEED)

    train_csv = train_csv_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val = int(len(train_csv) * VAL_RATIO)
    val_raw = train_csv[:n_val].copy()
    trn_raw = train_csv[n_val:].copy()
    trn_raw["tier"] = trn_raw["n_nouns"].apply(assign_tier)
    if is_main:
        val_raw.to_csv(CKPT_DIR / "_val_raw.csv", index=False)
        print(f"[v16] Processes={accelerator.num_processes}  train={len(trn_raw)}  val={len(val_raw)}")
        print(f"  tier 분포: {trn_raw['tier'].value_counts().to_dict()}")

    model, processor = load_model_and_processor()
    yes_id, no_id = get_yes_no_token_ids(processor)
    criterion = ListwiseSoftmaxLoss()

    train_ds = GroupedTemporalDataset(trn_raw)
    n_steps = (len(trn_raw) // GRAD_ACCUM) * EPOCHS
    n_warmup = int(n_steps * WARMUP_RATIO)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, n_warmup, n_steps)
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

    history = []  # (epoch, avg_loss, val_acc) — 나중에 시각화용
    best_val_acc = 0.0
    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        sampler = build_epoch_sampler(trn_raw, epoch, len(trn_raw))
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               collate_fn=collate_fn, num_workers=4, pin_memory=True)
        train_dl = accelerator.prepare(train_dl)

        if is_main:
            print(f"\n{'='*60}\nEpoch {epoch}/{EPOCHS}  tier weights={CURRICULUM_SCHEDULE.get(epoch)}\n{'='*60}")

        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for step, batch in enumerate(train_dl):
            with accelerator.accumulate(model):
                unwrapped = accelerator.unwrap_model(model)
                texts, imgs_list, group_offsets = [], [], []
                for grp_imgs, grp_sents, grp_dists in zip(batch["images"], batch["sentences"], batch["dists"]):
                    start = len(texts)
                    for imgs, sent in zip(grp_imgs, grp_sents):
                        msg = build_messages(imgs, sent)
                        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
                        texts.append(text)
                        imgs_list.append(imgs)
                    group_offsets.append((start, len(grp_imgs), grp_dists))

                logit_parts = []
                for bi in range(0, len(texts), TRAIN_MINIBATCH):
                    inp = processor(text=texts[bi:bi+TRAIN_MINIBATCH], images=imgs_list[bi:bi+TRAIN_MINIBATCH],
                                     return_tensors="pt", padding=True).to(device)
                    logit_parts.append(forward_logit(unwrapped, inp, yes_id, no_id))
                logits = torch.cat(logit_parts)

                ord_dists, ord_sizes = [], []
                for start, size, grp_dists in group_offsets:
                    ord_dists.extend(grp_dists)
                    ord_sizes.append(size)

                loss, per_dist_loss, weights = criterion(logits, ord_dists, ord_sizes)
                if not torch.isfinite(loss):
                    optimizer.zero_grad()
                    continue
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                epoch_loss += loss.item()
                n_batches += 1
                if is_main and global_step % LOGGING_STEPS == 0:
                    print(f"  step={global_step:5d}  loss={epoch_loss/n_batches:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

        accelerator.wait_for_everyone()
        if is_main:
            elapsed = (time.time() - t0) / 60
            avg_loss = epoch_loss / max(1, n_batches)
            print(f"\n[Epoch {epoch} 완료] {elapsed:.1f}분  avg_loss={avg_loss:.4f}")
            unwrapped = accelerator.unwrap_model(model)
            val_acc = val_exact_match(unwrapped, processor, val_raw, yes_id, no_id, device)
            history.append((epoch, avg_loss, val_acc))
            pd.DataFrame(history, columns=["epoch", "avg_loss", "val_acc"]).to_csv(LOG_DIR / "history.csv", index=False)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                unwrapped.save_pretrained(CKPT_DIR / CKPT_NAME)
                processor.save_pretrained(CKPT_DIR / CKPT_NAME)
                print(f"  * Best 저장 (val_acc={val_acc:.4f})")
            unwrapped.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
            processor.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
        accelerator.wait_for_everyone()

    if is_main:
        print(f"\n학습 완료. Best val_acc={best_val_acc:.4f}")

print("train_fn 정의 완료")

## 11. 학습 실행

`notebook_launcher`가 `NUM_GPUS`개 프로세스를 fork해서 `train_fn`을 각 GPU에서 실행한다
(리눅스 fork 방식이라 Jupyter/nbconvert 환경에서도 문제없이 동작).

In [ ]:
from accelerate import notebook_launcher

notebook_launcher(train_fn, num_processes=NUM_GPUS, mixed_precision="bf16")

## 12. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.read_csv(LOG_DIR / "history.csv")
print(history_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_df["epoch"], history_df["avg_loss"], marker="o")
axes[0].set_title("avg_loss per epoch"); axes[0].set_xlabel("epoch")
axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="o", color="green")
axes[1].set_title("val exact_match per epoch"); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig(LOG_DIR / "curves.png", dpi=120)
plt.show()

## 13. 추론 (24-permutation 전수조사) + 제출 파일 생성

In [ ]:
def run_inference(ckpt_name, out_name):
    from peft import PeftModel
    base_model = get_model_class(MODEL_PATH).from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, device_map={"": torch.device("cuda:0")}
    )
    processor = None
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = PeftModel.from_pretrained(base_model, str(CKPT_DIR / ckpt_name)).eval()
    yes_id, no_id = get_yes_no_token_ids(processor)

    test_df = pd.read_csv(DATA_DIR / "test.csv")
    submission = []
    for _, row in test_df.iterrows():
        sid, sentence = row["Id"], row["Sentence"]
        img_dir = DATA_DIR / "test" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        texts, imgs_list = [], []
        for perm in ALL_PERMS:
            inv = [0] * 4
            for k, t in enumerate(perm):
                inv[t - 1] = k
            imgs = [base_imgs[inv[t]] for t in range(4)]
            msg = build_messages(imgs, sentence)
            text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            texts.append(text)
            imgs_list.append(imgs)

        scores = []
        with torch.no_grad():
            for bi in range(0, len(texts), INFER_BATCH_SIZE):
                inp = processor(text=texts[bi:bi+INFER_BATCH_SIZE], images=imgs_list[bi:bi+INFER_BATCH_SIZE],
                                 return_tensors="pt", padding=True).to(model.device)
                s = forward_logit(model, inp, yes_id, no_id)
                scores.extend(s.cpu().tolist())
        best = max(zip(scores, [list(p) for p in ALL_PERMS]), key=lambda x: x[0])[1]
        submission.append({"Id": sid, "Answer": str(best)})

    out_path = PROJECT_ROOT / f"{out_name}.csv"
    pd.DataFrame(submission).to_csv(out_path, index=False)
    print(f"저장: {out_path} ({len(submission)}행)")
    del model, base_model
    torch.cuda.empty_cache()

run_inference(CKPT_NAME, "submission_v16_best")